In [1]:
from transformers import  T5Tokenizer, T5ForConditionalGeneration

In [2]:
import pandas as pd
train_data = pd.read_csv('datasets/samsum-train.csv')
val_data = pd.read_csv('datasets/samsum-validation.csv')

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [9]:
train_data.shape

(14731, 3)

In [10]:
val_data.shape

(818, 3)

In [11]:
train_data.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [12]:
train_data = train_data.dropna()

In [13]:
val_data.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [14]:
# train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)
# val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [15]:
train_data.shape, val_data.shape

((14731, 3), (818, 3))

In [16]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [17]:
import re
def clean_data(text):
    text = re.sub (r"\r\n"," ", text) # Lines
    text = re.sub(r"\s+",' ', text) # spaces
    text = re.sub(r"<.*?>", " " , text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [18]:
train_data.head()

,id,dialogue,summary
0,13818513,amanda: i baked cookies. do you want some? jer...,amanda baked cookies and will bring jerry some...
1,13728867,olivia: who are you voting for in this electio...,olivia and olivier are voting for liberals in ...
2,13681000,"tim: hi, what's up? kim: bad mood tbh, i was g...",kim may try the pomodoro technique recommended...
3,13730747,"edward: rachel, i think i'm in ove with bella....",edward thinks he is in love with bella. rachel...
4,13728094,sam: hey overheard rick say something sam: i d...,"sam is confused, because he overheard rick com..."


In [19]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [20]:
import torch
from torch.utils.data import Dataset

class SummaryDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        dialogue = self.data.iloc[index]['dialogue']
        summary = self.data.iloc[index]['summary']

        input_encoding = self.tokenizer(dialogue, padding='max_length', max_length=512, truncation=True, return_tensors = 'pt')
        target_encoding = self.tokenizer(summary, padding='max_length', max_length=150, truncation=True, return_tensors = 'pt')

        labels = target_encoding['input_ids'].squeeze()

        # Ignore padding tokens when calculating loss
        labels[labels == tokenizer.pad_token_id] = -100
        return {
            'input_ids':input_encoding['input_ids'].squeeze(),
            'attention_mask':input_encoding['attention_mask'].squeeze(),
            'labels':labels
        }
        

In [21]:
train_dataset = SummaryDataset(train_data, tokenizer)
val_dataset = SummaryDataset(val_data, tokenizer)

In [22]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [23]:
len(train_loader), len(val_loader)

(3683, 205)

In [27]:
# Device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Model
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

print("Device:", device)
print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Device: mps
Model device: mps:0


In [28]:
epochs = 4
for epoch in range(epochs):
    model.train()

    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        #remove old grad
        optimizer.zero_grad()

        #forward pass
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        #calculate loss
        loss = output.loss


        #backpropagration
        loss.backward()

        #update weights
        optimizer.step()


        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)

    print(f"epoch {epoch+1}/{epochs} training loss {avg_loss}")

epoch 1/4 training loss 1.7310669893905128
epoch 2/4 training loss 1.586659976865104
epoch 3/4 training loss 1.515329862123205
epoch 4/4 training loss 1.463604003441842


In [29]:
model.eval()

total_val_loss = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_val_loss += loss.item()

average_val_loss = total_val_loss / len(val_loader)

print(
    f"Validation Loss: {average_val_loss:.4f}"
)

Validation Loss: 1.4204


In [32]:
tokenizer.save_pretrained('./data')
model.save_pretrained('./data')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [33]:
tokenizer = T5Tokenizer.from_pretrained('./data/')
model = T5ForConditionalGeneration.from_pretrained('./data/')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [34]:
def summarize_dialogue(dialogue : str) -> str:
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [36]:
temp = """Artificial Intelligence (AI) is transforming the way people work, learn, and communicate. Many industries use AI to automate repetitive tasks, analyze large amounts of data, and improve decision-making. In healthcare, AI helps doctors detect diseases earlier by analyzing medical images and patient records. In education, AI-powered tools provide personalized learning experiences based on students' individual needs. Businesses also use AI chatbots to answer customer questions and provide support around the clock. However, AI also presents challenges, including data privacy concerns, job displacement, and the risk of biased decisions. To ensure responsible AI development, organizations must focus on transparency, fairness, and human supervision. Although AI cannot replace every human skill, it can significantly improve productivity when used responsibly."""
print(summarize_dialogue(temp))

artificial intelligence is transforming the way people work, learn, and communicate. many industries use ai to automate repetitive tasks, analyze large amounts of data, and improve decision-making. in healthcare, ai helps doctors detect diseases earlier by analyzing medical images and patient records. businesses also use ai chatbots to answer customer questions and provide support around the clock.
